# Wyciąganie cech wpływających na cenę z opisów ogłoszeń za pomocą OpenAI API

Kolumna `params` zawiera już dane ustrukturyzowane (piętro, typ budynku, ogrzewanie, ...), ale wiele informacji, od których zależy cena za m², występuje **tylko w opisie ogłoszenia**. Jakie to informacje? Zamiast decydować o tym samodzielnie, pozwalamy modelowi LLM **przejrzeć** ogłoszenia i zaproponować cechy.

Wyzwanie: jeśli dla każdego ogłoszenia poprosimy po prostu o *„cechy przydatne do predykcji ceny”*, każda odpowiedź będzie miała inne nazwy cech i nie zbudujemy z tego kolumn do modelu. Dlatego pracujemy w dwóch krokach:

1. **Odkrywanie** (jedno wywołanie z dłuższym „myśleniem”): LLM czyta próbkę ogłoszeń, widzi, jakie kolumny już mamy, i proponuje schemat cech: nazwy, typy, dozwolone wartości oraz uzasadnienie, dlaczego dana cecha powinna wpływać na cenę.
2. **Ekstrakcja** (jedno wywołanie na ogłoszenie, bez „myślenia”): zamieniamy propozycję na schemat i używamy **Structured Outputs**, dzięki czemu każde ogłoszenie zwraca dokładnie te same kolumny.

Po drodze oczyścisz opisy z HTML, będziesz śledzić tokeny, czas i koszt każdego wywołania, a wynik połączysz z danymi do swojego modelu ceny.

**Dane:** rozpakuj `real_estate_ads.csv.zip` i umieść `real_estate_ads.csv` obok tego notebooka.

In [ ]:
%pip install -q openai pandas pydantic python-dotenv

In [ ]:
import html
import importlib
import json
import os
import re
from typing import Literal, Optional

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, create_model

import openai_call_metrics
importlib.reload(openai_call_metrics)
from openai_call_metrics import (
    OpenAIMetricsSession,
    chat_completions_parse_with_metrics,
    format_openai_session_batch_totals,
)

In [ ]:
# Wczytuje OPENAI_API_KEY z pliku .env w folderze projektu (jeśli go używasz).
# .env jest w .gitignore - nigdy nie commituj swojego klucza.
load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "Nie znaleziono OPENAI_API_KEY. Zrób jedno z dwóch: "
        "(1) Utwórz plik .env obok tego notebooka z treścią: OPENAI_API_KEY=sk-... "
        "(2) Albo wyeksportuj klucz przed uruchomieniem Jupytera: export OPENAI_API_KEY=sk-... "
        "Notebooki w Cursor/VSCode często nie dziedziczą zmiennych wyeksportowanych w terminalu."
    )

In [ ]:
# Ceny za 1M tokenów, sprawdzone 2026-10-23 (developers.openai.com/api/docs/pricing).
# Oba kroki używają taniego modelu OpenAI do zadań na dużą skalę ($0.10 wejście / $0.50 wyjście).
# Odkrywanie (jedno wywołanie, reasoning_effort="medium") kosztuje ułamek centa.
DISCOVERY_MODEL = "gpt-6-luna"
MODEL = "gpt-6-luna"

client = OpenAI(api_key=api_key, max_retries=5)  # SDK ponawia zapytania przy limitach / chwilowych błędach
openai_metrics = OpenAIMetricsSession()

## 1. Wczytanie i czyszczenie danych

- Case dotyczy **rynku wtórnego**, więc filtrujemy po kolumnie `market` (ponad połowa ogłoszeń to rynek pierwotny).
- Bierzemy **losową próbkę** zamiast pierwszych wierszy, żeby kolejność w pliku nie zaburzała próbki.
- Opisy są w HTML. Usunięcie tagów oszczędza tokeny (= pieniądze) i daje modelowi czystszy tekst.

In [ ]:
DATA_PATH = "real_estate_ads.csv"
N_ADS = 5          # zacznij od małej liczby; zwiększ, gdy wyniki będą zadowalające
MAX_CHARS = 4000   # bardzo długie opisy przycinamy, żeby kontrolować koszt

data = pd.read_csv(DATA_PATH)
secondary = data[(data["market"] == "secondary") & data["description"].notna()]
sample = secondary.sample(n=N_ADS, random_state=42)
print(f"{len(data)} ogłoszeń łącznie, {len(secondary)} z rynku wtórnego, {len(sample)} w próbce")


def clean_description(raw: str) -> str:
    """Zamienia opis w HTML na zwykły tekst."""
    text = html.unescape(str(raw))
    text = re.sub(r"<br\s*/?>|</p>|</li>", "\n", text)  # zachowujemy podział na linie
    text = re.sub(r"<[^>]+>", " ", text)                 # usuwamy pozostałe tagi
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()[:MAX_CHARS]


print(clean_description(sample["description"].iloc[0]))

## 2. Parametry zapytania

### response_format (Structured Outputs)
Przekazanie klasy Pydantic jako `response_format` do `client.chat.completions.parse(...)` sprawia, że API zwraca JSON **gwarantowanie zgodny ze schematem**. SDK zamienia go na obiekt tej klasy (`message.parsed`), więc nie musimy sami parsować tekstu. Używamy tego w obu krokach: przy *propozycji* cech i przy *ekstrakcji* dla każdego ogłoszenia.

### reasoning_effort
Nowsze modele (GPT-5 i późniejsze) to modele *rozumujące*: zanim odpowiedzą, generują ukryte tokeny „myślenia”. Są one **rozliczane jak tokeny wyjściowe** i wliczają się do limitu długości.
- **Odkrywanie** to zadanie projektowe (porównać wiele ogłoszeń, zdecydować, co warto mieć jako kolumnę), więc pozwalamy modelowi pomyśleć: `reasoning_effort="medium"`.
- **Ekstrakcja** to odczytanie faktów z jednego ogłoszenia, więc używamy `reasoning_effort="none"`: taniej i szybciej. (Starsze modele `gpt-5-*` mają jako najniższą wartość `"minimal"`.)

Sprawdź `reasoning=` w podsumowaniu kosztów na końcu.

### Ograniczenie długości odpowiedzi: `max_tokens` vs `max_completion_tokens`
Ustala górny limit długości odpowiedzi w **tokenach** (fragmentach słów, nie słowach). Starsze modele używają `max_tokens`, nowsze **`max_completion_tokens`** (wysłanie `max_tokens` kończy się błędem 400). Ukryte rozumowanie wlicza się do limitu: jeśli dostajesz **pustą odpowiedź**, a liczba tokenów `out` jest wysoka, limit jest za niski. Przy ekstrakcji z wyłączonym rozumowaniem odpowiedź JSON potrzebuje tylko kilkuset tokenów, więc niski limit to tanie zabezpieczenie przed zbyt długą odpowiedzią.

### temperature i top_p
- `temperature` (0–2) steruje losowością: niższa = bardziej deterministyczne odpowiedzi.
- `top_p` (0–1) ogranicza losowanie do najbardziej prawdopodobnych tokenów (nucleus sampling).

Przy ekstrakcji chcemy deterministycznych odpowiedzi. Wiele modeli rozumujących akceptuje tylko wartości domyślne i w przeciwnym razie zwraca błąd 400, więc ich nie wysyłamy. **To, co API akceptuje, zależy od modelu**: sprawdź jego stronę na developers.openai.com.

## 3. Odkrywanie: LLM proponuje cechy

Pokazujemy modelowi próbkę ogłoszeń **oraz listę kolumn, które już mamy**, żeby szukał informacji *nowych*, a nie kopii `params`. Odpowiada w stałym formacie (`FeatureProposal`): lista cech z nazwą, typem, dozwolonymi wartościami i uzasadnieniem. To jedyne miejsce, w którym model „eksploruje”.

Prompt dla modelu jest po angielsku. Modele dobrze radzą sobie z polskimi ogłoszeniami także przy angielskich instrukcjach, a nazwy kolumn wychodzą wtedy po angielsku.

In [ ]:
N_DISCOVERY_ADS = 40          # wystarczająca różnorodność, by zobaczyć powtarzające się tematy; jedno wywołanie ~15-25k tokenów wejściowych
DISCOVERY_MAX_CHARS = 1500    # krótsze fragmenty, żeby 40 ogłoszeń zmieściło się w jednym prompcie

# Kolumny, które już mamy: klucze z `params` plus główne kolumny zbioru danych.
params_keys = sorted({
    kv.split("<=>")[0]
    for p in secondary["params"].dropna()
    for kv in p.split("<br>") if "<=>" in kv
})
known_columns = ["m (area)", "no_rooms", "price", "location (map_lat, map_lon, district)"] + params_keys
print("Już dostępne:", ", ".join(known_columns))

discovery_sample = secondary.drop(sample.index).sample(n=N_DISCOVERY_ADS, random_state=0)
ads_text = "\n\n".join(
    f"--- AD {i + 1} ---\n{clean_description(d)[:DISCOVERY_MAX_CHARS]}"
    for i, d in enumerate(discovery_sample["description"])
)


class ProposedFeature(BaseModel):
    name: str = Field(description="Column name in English snake_case, e.g. 'has_garden'.")
    type: Literal["boolean", "integer", "number", "category"]
    categories: list[str] = Field(description="Allowed values (snake_case) if type is 'category', otherwise an empty list.")
    description: str = Field(description="What the feature means and how to recognise it in a Polish ad.")
    why_price_relevant: str = Field(description="One sentence: why it should affect the price per m2.")


class FeatureProposal(BaseModel):
    features: list[ProposedFeature]


DISCOVERY_PROMPT = f"""You are helping to build a model that predicts the price per m2 of apartments
on the secondary market in Poznań. Below are real ads (in Polish).

Propose 10-20 features that:
- can be read from the description text,
- are likely to affect the price per m2,
- appear in a reasonable share of ads (not one-off details),
- are NOT already available as columns: {", ".join(known_columns)}.

Prefer simple types (boolean, integer, a short list of categories). Do not add an 'unknown'
category: a missing value will be returned as null."""

response = chat_completions_parse_with_metrics(
    client,
    session=openai_metrics,
    model=DISCOVERY_MODEL,
    messages=[
        {"role": "system", "content": DISCOVERY_PROMPT},
        {"role": "user", "content": ads_text},
    ],
    response_format=FeatureProposal,
    reasoning_effort="medium",
    max_completion_tokens=16_000,  # miejsce na ukryte rozumowanie + odpowiedź
)
message = response.choices[0].message
if message.refusal:
    raise ValueError(f"Model odmówił odpowiedzi: {message.refusal}")
proposal = message.parsed

pd.DataFrame([f.model_dump() for f in proposal.features])

### Przegląd propozycji (człowiek w pętli)

Propozycja zostaje zapisana do `feature_schema.json`. Przeczytaj ją krytycznie: czy te cechy naprawdę wpływają na cenę? Czy któreś nie dublują istniejących kolumn? Możesz **edytować plik** (usuwać, zmieniać nazwy, dodawać cechy) i uruchomić ponownie kolejną komórkę. Zawsze wczytuje ona plik, więc ekstrakcja użyje Twojej poprawionej wersji. Dzięki temu ekstrakcja jest też powtarzalna: ponowne odkrywanie może dać nieco inną propozycję.

In [ ]:
SCHEMA_PATH = "feature_schema.json"
with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(proposal.model_dump(), f, indent=2, ensure_ascii=False)

### Zamiana propozycji na schemat

`create_model` buduje z pliku JSON klasę Pydantic, taką samą, jaką w innym przypadku napisalibyśmy ręcznie:
- `Literal[...]` = model musi wybrać jedną z podanych wartości (cecha kategoryczna).
- `Optional[...]` = model może zwrócić `null`, gdy ogłoszenie o czymś nie wspomina. To ważne: chcemy *braku danych*, a nie zgadywania.
- `description` każdego pola jest wysyłany do modelu, więc działa też jak instrukcja.

In [ ]:
TYPE_MAP = {"boolean": bool, "integer": int, "number": float}


def build_schema(path: str) -> type[BaseModel]:
    with open(path, encoding="utf-8") as f:
        features = FeatureProposal.model_validate(json.load(f)).features
    fields = {}
    for feat in features:
        name = re.sub(r"\W+", "_", feat.name.strip().lower()).strip("_")  # poprawny identyfikator Pythona
        if not name or name[0].isdigit() or name.startswith("model_") or name in fields:
            print(f"pomijam cechę z niepoprawną lub powtórzoną nazwą: {feat.name!r}")
            continue
        if feat.type == "category" and feat.categories:
            py_type = Literal[tuple(feat.categories)]
        else:
            py_type = TYPE_MAP.get(feat.type, str)
        fields[name] = (Optional[py_type], Field(description=feat.description))
    return create_model("ApartmentFeatures", **fields)


ApartmentFeatures = build_schema(SCHEMA_PATH)
print(json.dumps(ApartmentFeatures.model_json_schema(), indent=2, ensure_ascii=False)[:2000])

## 4. Ekstrakcja: wypełnienie schematu dla każdego ogłoszenia

In [ ]:
SYSTEM_PROMPT = """You are a data extraction assistant for real estate listings.
You receive a Polish apartment-for-sale ad from Poznań (secondary market).
Extract the requested fields using ONLY information stated in the ad.
If something is not mentioned, return null - never guess."""


def extract_features(description: str) -> ApartmentFeatures:
    response = chat_completions_parse_with_metrics(
        client,
        session=openai_metrics,
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": clean_description(description)},
        ],
        response_format=ApartmentFeatures,
        reasoning_effort="none",
        max_completion_tokens=1_000,
    )
    message = response.choices[0].message
    if message.refusal:  # model może odmówić zamiast odpowiedzieć - obsługujemy to jawnie
        raise ValueError(f"Model odmówił odpowiedzi: {message.refusal}")
    return message.parsed

In [ ]:
# Uruchom ponownie komórkę z client + openai_metrics, jeśli chcesz liczyć koszt i czas od zera.
rows = []
for ad_id, description in zip(sample["id"], sample["description"]):
    try:
        features = extract_features(description)
    except Exception as e:  # jedno problematyczne ogłoszenie nie powinno zatrzymać całej pętli
        print(f"id={ad_id} błąd: {e}")
        continue
    rows.append({"id": ad_id, **features.model_dump()})

features_df = pd.DataFrame(rows)
features_df

## 5. Koszt i czas

Pierwsze wywołanie to odkrywanie, pozostałe to ekstrakcje. `out` zawiera ukryte tokeny `reasoning`. Ostatnia linia szacuje koszt ekstrakcji dla wszystkich ogłoszeń z rynku wtórnego. Warto to sprawdzić **zanim** uruchomisz cały zbiór danych. Odkrywanie to koszt jednorazowy.

In [ ]:
print(format_openai_session_batch_totals(openai_metrics))
print()
print(openai_metrics.format_summary())

extraction_calls = [c for c in openai_metrics.calls if c.model.startswith(MODEL)]
if extraction_calls and all(c.cost_usd is not None for c in extraction_calls):
    per_ad = sum(c.cost_usd for c in extraction_calls) / len(extraction_calls)
    print(f"\nŚredni koszt ekstrakcji na ogłoszenie: ${per_ad:.6f}  ->  wszystkie {len(secondary)} ogłoszenia z rynku wtórnego: ~${per_ad * len(secondary):.2f}")

## 6. Czy propozycja jest dobra?

LLM zaproponował cechy, które *jego zdaniem* mają znaczenie. Dwa szybkie sprawdzenia:
- **Stopień wypełnienia:** jak często cecha faktycznie pojawia się w ogłoszeniach? Cecha, która ma `null` w 95% ogłoszeń, niewiele pomoże.
- **Różnice w cenie:** czy cena za m² różni się między wartościami cechy? Przy 5 ogłoszeniach to tylko test poprawności działania; uruchom kilkaset, żeby zobaczyć prawdziwe różnice.

Nowe kolumny można połączyć z pełnym zbiorem danych po `id` i użyć w modelu ceny. Wartości logiczne z `null` stają się trzecią kategorią („nie wspomniano”), która często sama w sobie niesie informację.

In [ ]:
feature_cols = list(ApartmentFeatures.model_fields)
features_df[feature_cols].notna().mean().sort_values(ascending=False).round(2).rename("fill_rate")

In [ ]:
merged = sample[["id", "price_per_m", "m", "no_rooms"]].merge(features_df, on="id")

for col in feature_cols:
    if merged[col].nunique() <= 6:  # cechy logiczne i kategoryczne
        print(merged.groupby(col, dropna=False)["price_per_m"].agg(["count", "mean"]).round(0), "\n")

## Ćwiczenia

1. **Sprawdź jakość.** Przeczytaj samodzielnie 10–20 opisów i porównaj je z wyciągniętymi wartościami. Gdzie model się myli? Czy lepszy `description` cechy w `feature_schema.json` albo lepszy prompt systemowy to naprawi?
2. **Przejrzyj propozycję.** Które z zaproponowanych cech byś usunął/usunęła, połączył(a) albo dodał(a)? Edytuj `feature_schema.json` i uruchom notebook ponownie od komórki „Zamiana propozycji na schemat” (nie trzeba ponownie wywoływać odkrywania).
3. **Zbadaj stabilność odkrywania.** Uruchom odkrywanie ponownie z innym `random_state` albo większą liczbą ogłoszeń. Na ile stabilna jest propozycja? Które cechy pojawiają się za każdym razem?
4. **Porównaj modele.** Uruchom ekstrakcję z `gpt-5.6-luna` (poprzednia generacja, ok. 2× droższa). Czy wyniki się różnią?
5. **Zmierz wartość.** Wytrenuj model ceny za m² z cechami z LLM i bez nich. Które cechy zmniejszają błąd (feature importance)?
6. **Skaluj tanio.** Przy tysiącach ogłoszeń zapisuj wyniki do pliku, żeby nigdy nie płacić dwa razy za to samo ogłoszenie, i zobacz [Batch API](https://developers.openai.com/api/docs/guides/batch) (50% taniej, wyniki w ciągu 24 h).

**Uwaga o prywatności:** opisy mogą zawierać imiona i nazwiska agentów. Wszystko, co jest w prompcie, trafia do OpenAI.